In [6]:
import pandas as pd

# Create a DataFrame
data = pd.DataFrame({
    "Experience": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    "Salary": [25000, 28000, 32000, 35000, 40000,
               45000, 50000, 55000, 60000, 65000,
               70000, 75000, 80000, 85000, 90000]
})

print(data)

# Define feature and target
X = data[["Experience"]]
y = data["Salary"]

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)

    Experience  Salary
0            1   25000
1            2   28000
2            3   32000
3            4   35000
4            5   40000
5            6   45000
6            7   50000
7            8   55000
8            9   60000
9           10   65000
10          11   70000
11          12   75000
12          13   80000
13          14   85000
14          15   90000

Feature shape: (15, 1)
Target shape: (15,)


In [7]:

import numpy as np
import pandas as pd

# Set random seed
rng = np.random.default_rng(7)

# Number of students
n = 20

# Create student dataset
data = pd.DataFrame({
    "hours": rng.integers(1, 11, n),
    "sleep": rng.integers(4, 10, n),
    "attendance": rng.integers(60, 101, n)
})

# Generate random noise
noise = rng.normal(0, 4, n)

# Calculate marks
data["marks"] = (
    10
    + 4 * data["hours"]
    + 2 * data["sleep"]
    + 0.5 * data["attendance"]
    + noise
).round()

# Keep marks within 0 to 100
data["marks"] = data["marks"].clip(0, 100).astype(int)

# Display dataset
print(data)


    hours  sleep  attendance  marks
0      10      8          79     99
1       7      5          68     80
2       7      6          94     93
3       9      5          66     86
4       6      8          95    100
5       8      5          85     91
6       9      9          64     96
7       3      6          61     68
8       1      6          78     63
9       4      7          61     70
10      3      7          65     69
11      9      7          81    100
12     10      7          99    100
13      1      9          79     72
14      5      8          93     98
15      9      8          97    100
16      2      8          93     84
17      8      7          85     99
18      2      6          78     66
19      5      9          81     97


In [8]:
data = pd.DataFrame({
    "hours": rng.integers(1, 11, n),
    "sleep": rng.integers(4, 10, n),
    "attendance": rng.integers(60, 101, n)})
noise = rng.normal(0, 4, n)
data["marks"] = (10 + 4 * data["hours"] + 2 * data["sleep"] + 0.5 * data["attendance"] + noise).round()
data["marks"] = data["marks"].clip(upper=100).astype(int)

In [9]:
#split the data

from sklearn.model_selection import(
    train_test_split)
from sklearn.linear_model import(
    LinearRegression)

train,test=train_test_split(
    data,test_size=0.25,random_state=42)
y_train=train["marks"]
y_test=test["marks"]
print(len(train),len(test))

15 5


In [10]:
#fit a simple model on hours

from sklearn.metrics import(r2_score,mean_absolute_error)

simple=LinearRegression()
simple.fit(train[["hours"]],y_train)
pred1=simple.predict(test[["hours"]])
print("slope:",simple.coef_.round(2))
print("intercept:",simple.intercept_.round(2))
r2 = r2_score(y_test,pred1)
print("test R2:",round(r2,3))

slope: [2.88]
intercept: 67.43
test R2: 0.23


In [11]:
m,c=np.polyfit(train["hours"],y_train, 1)
print(round(m,2),round(c,2))

2.88 67.43


In [12]:
#look at the residuals

resid =y_test-pred1
print(resid.round(1).head(5).tolist())
print("mean:",round(resid.mean(),2))

[-9.6, -6.7, 3.8, -1.7, 9.9]
mean: -0.86


In [13]:

from sklearn.linear_model import LinearRegression
import pandas as pd

# Train with three features
feats = ["hours", "sleep", "attendance"]

multi = LinearRegression()
multi.fit(train[feats], y_train)

# Display coefficients
coefs = pd.Series(multi.coef_, index=feats)

print("Coefficients:")
print(coefs.round(2))

print("Intercept:", round(multi.intercept_, 2))


Coefficients:
hours         3.13
sleep         1.99
attendance    0.57
dtype: float64
Intercept: 6.55


In [14]:
#simple vs multiple

pred2 = multi.predict(test[feats])
result = {"Simple": pred1, "Multiple": pred2}
for name, p in result.items():
    r2 = r2_score(y_test, p)
    mae = mean_absolute_error(y_test, p)
    print(f"{name} R2: {round(r2, 3)}, MAE: {round(mae, 3)}")


Simple R2: 0.23, MAE: 6.341
Multiple R2: 0.744, MAE: 3.94


In [15]:
#predict for a new student

student=pd.DataFrame({
    "hours": [6],"sleep":[8],
    "attendance":[90]})
print(multi.predict(student).round(1))

[92.4]


In [16]:
from sklearn.preprocessing import (StandardScaler)

scaler=StandardScaler()
Xs_train=scaler.fit_transform(train[feats])
Xs_test=scaler.transform(train[feats])
print(Xs_train.mean(axis=0).round(2)+0)
print(Xs_train.std(axis=0).round(2))

[0. 0. 0.]
[1. 1. 1.]


In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import pandas as pd

# Train model using scaled features
scaled = LinearRegression()
scaled.fit(Xs_train, y_train)

# Display coefficients
sc = pd.Series(scaled.coef_, index=feats)
print("Scaled Coefficients:")
print(sc.round(2))

# Predict test data
pred3 = scaled.predict(Xs_test)

# Evaluate model
r2 = r2_score(y_test, pred3)
print("Test R2:", round(r2, 3))


Scaled Coefficients:
hours         6.43
sleep         3.54
attendance    6.74
dtype: float64


ValueError: Found input variables with inconsistent numbers of samples: [5, 15]

In [ ]:
from sklearn.pipeline import make_pipeline
pipe = make_pipeline(StandardScaler(), LinearRegression())
pipe.fit(train[feats], y_train)
r2 = pipe.score(test[feats], y_test)
print("Test R2:", round(r2, 3))

Test R2: 0.744


In [ ]:
print(data.corr()["marks"].round(2))


hours         0.65
sleep         0.30
attendance    0.42
marks         1.00
Name: marks, dtype: float64


In [ ]:
#when the data curves4


curve=pd.DataFrame({
    "x":[1,2,3,4,5,6,7,8,9,10],
    "y":[2,5,10,17,26,37,50,65,82,101]})
cx,cy=curve[["x"]],curve["y"]
line=LinearRegression().fit(cx,cy)
print("line R2",round(line.score(cx,cy),3))

line R2 0.95


In [19]:

import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression

# Features and target
feats = ["hours", "sleep", "attendance"]
X = data[feats]
y = data["marks"]

# Create and train the multiple regression model
multi = LinearRegression()
multi.fit(X, y)

# Save the trained model
joblib.dump(multi, "marks_multi.pkl")

# Load the saved model
loaded = joblib.load("marks_multi.pkl")

# Create new student data
student = pd.DataFrame({
    "hours": [6],
    "sleep": [8],
    "attendance": [90]
})

# Predict marks
print("Predicted Marks:", loaded.predict(student).round(1))


Predicted Marks: [91.8]
